# FreshRetailNet — Foundation Model Forecasting with MMF (Serverless GPU)

This notebook runs **foundation time series models** on the FreshRetailNet demand data
using the [Many Model Forecasting (MMF)](https://github.com/databricks-industry-solutions/many-model-forecasting) framework
on **serverless GPU** compute.

### Prerequisites
- Run the `fresh_retail_net_mmf` notebook first to create `mmf.fresh_retail_net.demand_train` and `mmf.fresh_retail_net.demand_score`
- Attach this notebook to **Serverless** compute with **A10 GPU** accelerator and **Environment version 5+**

### Foundation Models
We run the following pre-trained foundation models (no training required — zero-shot forecasting):
- **Chronos-Bolt** (Tiny, Mini, Small, Base) — Amazon's efficient time series foundation model
- **Chronos 2** (Base, Small, Synth) — Amazon's next-gen foundation model
- **TimesFM 2.5** — Google's 200M parameter time series foundation model

## 1. Install Dependencies

In [ ]:
%pip install "mmf_sa[foundation] @ git+https://github.com/databricks-industry-solutions/many-model-forecasting.git" hf_transfer --quiet
dbutils.library.restartPython()

## 2. Configuration

In [ ]:
import logging
import uuid

# Suppress noisy loggers on serverless
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.WARNING)
logging.getLogger("py4j.java_gateway").setLevel(logging.WARNING)

catalog = "mmf"
db = "fresh_retail_net"
user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]

# Verify tables exist
train_count = spark.table(f"{catalog}.{db}.demand_train").count()
score_count = spark.table(f"{catalog}.{db}.demand_score").count()
print(f"Train rows: {train_count:,}  |  Score rows: {score_count:,}")

## 3. Define Foundation Models

In [ ]:
active_models = [
    "ChronosBoltBase",
    "Chronos2",
    "TimesFM_2_5_200m",
]

print(f"Models to run: {len(active_models)}")
for m in active_models:
    print(f"  - {m}")

## 4. Run Foundation Model Forecasts

On serverless GPU, we run **one model at a time** in a loop because Spark Connect Python workers
are CPU-only even when the driver has a GPU. The `serverless=True` flag enables driver-only
prediction path.

In [ ]:
from mmf_sa import run_forecast

run_id = str(uuid.uuid4())
prediction_length = 7  # 7-day forecast horizon (matches eval split)

for model in active_models:
    print(f"\n{'='*60}")
    print(f"Running {model} (run_id={run_id})")
    print(f"{'='*60}")
    run_forecast(
        spark=spark,
        train_data=f"{catalog}.{db}.demand_train",
        scoring_data=f"{catalog}.{db}.demand_score",
        scoring_output=f"{catalog}.{db}.scoring_output",
        evaluation_output=f"{catalog}.{db}.evaluation_output",
        model_output=f"{catalog}.{db}",
        group_id="unique_id",
        date_col="ds",
        target="y",
        freq="D",
        prediction_length=prediction_length,
        backtest_length=21,
        stride=7,
        metric="smape",
        train_predict_ratio=2,
        data_quality_check=False,
        resample=False,
        active_models=[model],
        dynamic_future_numerical = ["discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"], 
        dynamic_future_categorical =["holiday_flag", "activity_flag"],
        experiment_path=f"/Users/{user}/mmf/fresh_retail_net",
        use_case_name="fresh_retail_net",
        run_id=run_id,
        accelerator="gpu",
        serverless=True,
    )

## 5. Review Evaluation Results

In [ ]:
%sql
SELECT
  model,
  COUNT(DISTINCT unique_id) AS num_series,
  ROUND(AVG(metric_value), 4) AS avg_smape,
  ROUND(PERCENTILE(metric_value, 0.5), 4) AS median_smape
FROM mmf.fresh_retail_net.evaluation_output
GROUP BY model
ORDER BY avg_smape

## 6. Review Scoring Output (Future Predictions)

In [ ]:
%sql
SELECT
  model,
  COUNT(DISTINCT unique_id) AS num_series,
  COUNT(*) AS total_predictions,
  MIN(ds) AS min_date,
  MAX(ds) AS max_date
FROM mmf.fresh_retail_net.scoring_output
GROUP BY model
ORDER BY model

In [ ]:
%sql
-- Sample predictions for the top-demand series across all models
WITH top_series AS (
  SELECT unique_id
  FROM mmf.fresh_retail_net.demand_train
  GROUP BY unique_id
  ORDER BY SUM(y) DESC
  LIMIT 1
)
SELECT s.model, s.ds, s.unique_id, s.y AS predicted
FROM mmf.fresh_retail_net.scoring_output s
JOIN top_series t ON s.unique_id = t.unique_id
ORDER BY s.model, s.ds